In [ ]:
import os
import sys
import numpy as np
import pyvista as pv
import pyacvd
import open3d as o3d


sys.path.append(os.path.abspath(".."))
from SUPORT_def_deformation import *

In [ ]:
direct = "./"
infolder_total = os.listdir(direct)
print(infolder_total)




Cline = pv.read("centerline.vtk")
print("centerline loaded")

# STATE selects which deformed phantom is reconstructed. Change it to def2 or
# def3 and re-run this notebook and 3_score to reproduce the other two; that is
# how the three published Dice and radius values were obtained.
#
# The names were ATAA_sint1.stl and ATAA_sint_def11.stl, which no notebook here
# writes — the author's copies of those were a denser test of the same shapes,
# about forty times as many points. He confirmed in August 2026 that the point
# count does not affect the reconstruction, so these read what 1_build_phantom
# produces and the three notebooks now run in order.
STATE = 'def1'
load_name_mesh_load = ['ATAA_sint.stl', f'ATAA_sint_{STATE}.stl']
meshes = []

for file in load_name_mesh_load:
    mesh_r = pv.read(file).clean().fill_holes(50)
    clus = pyacvd.Clustering(mesh_r)
    clus.cluster(10000)
    mesh_rem = clus.create_mesh()
    mesh_rem.fill_holes(1, inplace=True)
    meshes.append(mesh_rem)

original_0 = meshes[0].copy()

Cline = pv.read("centerline.vtk")

plot_mesh(mesh1=meshes[0],mesh2=Cline)



In [ ]:
file_path = "cuts_posit.txt"
data = read_cuts_posit(file_path)
plane_info_0 = np.array(data[0])
plane_info_arc = np.array(data[1])
print(plane_info_0)
print(plane_info_arc)



## light inlet


In [ ]:
def preprocess_point_cloud(pcd, voxel_size):
    pcd_down = pcd.voxel_down_sample(voxel_size)
    radius_normal = voxel_size * 2
    pcd_down.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=30)
    )
    radius_feature = voxel_size * 5
    fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down,
        o3d.geometry.KDTreeSearchParamHybrid(radius=radius_feature, max_nn=100)
    )
    return pcd_down, fpfh


def align_point_clouds(source_points, target_points, voxel_size=0.9):
    source = o3d.geometry.PointCloud()
    source.points = o3d.utility.Vector3dVector(source_points)
    target = o3d.geometry.PointCloud()
    target.points = o3d.utility.Vector3dVector(target_points)
    source_down, source_fpfh = preprocess_point_cloud(source, voxel_size)
    target_down, target_fpfh = preprocess_point_cloud(target, voxel_size)

    ransac_result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        source_down, target_down, source_fpfh, target_fpfh,
        mutual_filter=True,
        max_correspondence_distance=voxel_size * 2,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
        ransac_n=4,
        checkers=[
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(voxel_size * 2)
        ],
        criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(4000000, 500)
    )

    # Estimate normals for full resolution clouds for ICP
    source.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 2, max_nn=30))
    target.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 2, max_nn=30))
    
    icp_result = o3d.pipelines.registration.registration_icp(
        source, target,
        max_correspondence_distance=voxel_size * 1.5,
        init=ransac_result.transformation,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPlane()
    )

    return icp_result.transformation

size_cube = 70
cut_cube = pv.Cube(center=Cline.points[0], x_length=size_cube, y_length=size_cube, z_length=size_cube)
meshes_to_align = [mesh.clip_box(cut_cube, invert=False) for mesh in meshes]

source_points = np.array(meshes_to_align[0].points)
target_points = np.array(meshes_to_align[1].points)
transformation_matrix = align_point_clouds(source_points, target_points)
inv_matrix = np.linalg.inv(transformation_matrix)



In [ ]:
plotter = pv.Plotter()
plotter.add_mesh(meshes_to_align[0], color="blue", opacity=0.2)
plotter.add_mesh(meshes_to_align[1].transform(inv_matrix, inplace=False), color="red", opacity=0.2)
plotter.add_mesh(cut_cube, color='black', opacity=0.2)
plotter.show()

In [ ]:
meshes[0].clip(origin=plane_info_0[0], normal=plane_info_0[1],invert=False,inplace=True).connectivity('largest')
meshes[1].transform(inv_matrix, inplace=True).clip(origin=plane_info_0[0], normal=plane_info_0[1],invert=False,inplace=True).connectivity('largest')

Cline.clip(origin=plane_info_0[0], normal=plane_info_0[1],invert=False,inplace=True).connectivity('largest')
plot_mesh(mesh1=meshes[0],mesh2=meshes[1])

In [ ]:
def cut_ring(mesh, cpoint, normal1,size=0.5):
    C_dist_1 = mesh.clip(normal=normal1, origin=cpoint + normal1 * size, invert=True)
    C_dist = C_dist_1.clip(normal=normal1, origin=cpoint - normal1 * size, invert=False)
    if len(C_dist.points) == 0:
        print("error")
        return None
    return C_dist

In [ ]:
def calculate_points_ref_valid (rings,vector_normal):
    
    vectors=calculate_vectors(vector_normal)
    point_matching=[]

    for ring in (rings):
            ring_sec=[]
            for vector in vectors:
                point_find,_=find_optimal_point_intrs(ring, ring.center_of_mass(),vector)
                if point_find is not None:
                        ring_sec.append(point_find)

            point_matching.append(np.array(ring_sec))
    return vectors,point_matching

In [ ]:
ring_inlet_list_orig=[]
plotter = pv.Plotter(notebook=True)
plotter.add_mesh(meshes[0], color="blue", opacity=0.2)

for num_mesh,mesh in enumerate(meshes):
        ring_T =cut_ring(mesh, plane_info_0[0], plane_info_0[1])
        ring_inlet_list_orig.append(ring_T)

        
for i in ring_inlet_list_orig:
    plotter.add_mesh(i, color="red", opacity=1)


vectors_inlet,point_matching_inlet_T=calculate_points_ref_valid(ring_inlet_list_orig,plane_info_0[1])
print(vectors_inlet)
number_rings_points=(np.mean([len(point) for point in point_matching_inlet_T]))
for i,mesh in enumerate(meshes):
        for vector in vectors_inlet:
            plotter.add_lines(np.array([plane_info_0[0],plane_info_0[0]+vector*10]),color="b",width=2)
            plotter.add_points(point_matching_inlet_T[i],color="green",point_size=10)
        
        
        
plotter.show() 

In [ ]:

# reset trnasformation
plane_info=[None]*len(meshes)
point_matching_inlet=[None]*len(point_matching_inlet_T)


inverse_matrix = transformation_matrix


plane_info[0]=plane_info_0
plane_info[1]=trasform_plane(plane_info_0,inverse_matrix)
point_matching_inlet[0]=point_matching_inlet_T[0]
point_matching_inlet[1]=trasform_points(point_matching_inlet_T[1],inverse_matrix)
meshes[1]=mesh.transform(inverse_matrix, inplace=True)

deform_0 = meshes[1].copy()
nam2e=load_name_mesh_load[1]+"_def0_cut.vtp"
deform_0.save(nam2e)


In [ ]:
plotter=pv.Plotter() 
plotter.add_mesh(meshes[0], color="blue", opacity=0.2)  
plotter.add_mesh(meshes[1], color="red", opacity=0.2)  
# plotter.add_mesh(meshes[1].transform(inverse_matrix, inplace=False),color='red', opacity=0.2)
plotter.add_mesh(original_0,opacity=1)
plotter.show(jupyter_backend='static')

## Start Arc

In [ ]:
# meshes_arc=[None]*len(meshes)
# for num_mesh,mesh in enumerate(meshes):

#     mesh_T=cut_ring(mesh, plane_info_arc[0]-plane_info_arc[1]*5, plane_info_arc[1],size=0.5)
#     meshes_arc[num_mesh]=select_closest(mesh_T, plane_info_arc[0])
    
    
# plotter=pv.Plotter(notebook=False) 
# plotter.add_mesh(meshes[0], color="blue", opacity=0.2)   
# for num_mesh,mesh in enumerate(meshes):

#     rand=np.random.randint(0,5)
#     plotter.add_mesh(meshes_arc[num_mesh], color=cores[rand],opacity=0.5)
#     # plotter.add_text("Mesh "+str(num_mesh), font_size=10, position='upper_left')
#     plotter.camera_position = 'yz'
# plotter.show(jupyter_backend='static')

In [ ]:
def calculate_vectors(vector_main,number_vectors=16,first_vector=None):
    rotation_angle = 2 * np.pi / number_vectors
    new_vectors=[]
    if first_vector is not None: 
        projection_main = np.dot(first_vector, vector_main) * vector_main
        vectors = first_vector - projection_main
    else:
            perp_vector = np.cross(vector_main, [0, 0, 1])# z-axis as arbitrary vector
            vectors =np.array(  perp_vector / np.linalg.norm(perp_vector)) # Normalize
    
    for i in range(0, number_vectors):
        rotated_vector = rodrigues_rotation(vectors, vector_main, rotation_angle * i)
        rotated_vector = rotated_vector / np.linalg.norm(rotated_vector)
        new_vectors.append(rotated_vector)
    return new_vectors




def calculate_points_dif_valid (ring,center_temp,vector_normal,vectors=None,number_vectors=16):

    if vectors is None:vectors=calculate_vectors(vector_normal,number_vectors,first_vector=np.array([0,0,1]))
    
    ring_sec=[]
    result_t=[]
    for vector in vectors:
        points_finded,t=find_optimal_point_intrs(ring,center_temp,vector)
        result_t.append(t)
        if points_finded is not None:
            ring_sec.append(points_finded)
    return vectors,ring_sec,result_t

In [ ]:
ring_arc_list_orig=[]
point_matching_arc_temp=[]
vectors_arc=None
ring_Tval=[]
center_arc_ring= plane_info_arc[0]-plane_info_arc[1]

Cline.clip(origin=center_arc_ring, normal=plane_info_arc[1],invert=True,inplace=True).connectivity('largest')

plotter = pv.Plotter(notebook=True)
plotter.add_mesh(meshes[0],opacity=0.3)
plotter.add_mesh(Cline, color='black', point_size=5)

for num_mesh,mesh in enumerate(meshes):
    ring=cut_ring(mesh,center_arc_ring, plane_info_arc[1],size=0.5)
    
    ring_arc_list_orig.append(ring)
    vectors_arc,point_matching_arc_temp2,ring_T=calculate_points_dif_valid (ring,center_arc_ring,plane_info_arc[1]) 

    
    
    plotter.add_mesh(ring, color="green",opacity=1)
    for i,vec in enumerate(vectors_arc):
        plotter.add_lines(np.array([center_arc_ring, center_arc_ring+vec*10]), color='blue')
        plotter.add_points(point_matching_arc_temp2[i],color="yellow",point_size=10)

    ring_Tval.append(ring_T)
    point_matching_arc_temp.append(point_matching_arc_temp2)
plotter.show(jupyter_backend='static')

In [ ]:
point_matching_arc =point_matching_arc_temp#correct_t(point_matching_arc_temp,ring_Tval, center_arc_ring, vectors_arc)

print(point_matching_arc_temp)

In [ ]:

plotter=pv.Plotter(notebook=True)


plotter.add_mesh(meshes[1], color="w",opacity=0.1)


for j in [0,1]:
    for i in point_matching_arc[j]:
        plotter.add_points(i, color="r")
    for i in point_matching_inlet[j]:
        plotter.add_points(i, color="r")

plotter.show(jupyter_backend='static')

In [ ]:

for i in range(len(point_matching_arc)):
    if len(point_matching_inlet[i]) !=len(point_matching_arc[i]):
        print(i,"point_matching_inlet",len(point_matching_inlet[i]))
        print(i,"point_matching_arc",len(point_matching_arc[i]))
original_points=np.concatenate((point_matching_inlet,point_matching_arc),axis=1)

print(len(original_points[0]))
meshes_new,deformation_init=rbf_calculation(meshes,original_points,original_points[0])

In [ ]:
number_rings =int(len(Cline.points)*0.5)

ring_centers=[None]*(len(meshes))


for num,mesh in enumerate(meshes):
    Cline_len=centerline_length(Cline.points,plane_info[num][0])
    ring_centers_per_mesh=[]
    
    for i in range(0,number_rings-1):
        interval_Cline=[[0,0,0],[0,0,0]]
        if i==0 : Cline_posit=Cline_len
        else: Cline_posit=(Cline_len / number_rings) * (i)
        
        interval_Cline[0],interval_Cline[1]=calc_cut_posit(Cline.points,Cline_posit)
        if interval_Cline[0] is not None: ring_centers_per_mesh.append(interval_Cline)

    
    ring_centers[num]=(ring_centers_per_mesh)
deformation_secod=[[]]*len(meshes)
meshes_new_new=[None]*len(meshes)

target_points=[]
origin_points=[]
for num in range(len(meshes)):

    # if num==7:plotter=pv.Plotter(notebook=False)
    point_matching_list_orig,point_matching_list_orig_new = [original_points[num]],[original_points[num]]
    # if num==7:plotter.add_mesh(meshes[num], color="white",opacity=0.2)
    # if num==7:plotter.add_mesh(meshes_new[num], color="red",opacity=0.2)
    # if num==7:plotter.add_mesh(Cline, color="white",opacity=0.2)
    
    for ring_num in range(len(ring_centers[num])):    
        ring=cut_ring(meshes[num], ring_centers[num][ring_num][0], ring_centers[num][ring_num][1])
        ring_new=cut_ring(meshes_new[num], ring_centers[num][ring_num][0], ring_centers[num][ring_num][1])
        if ring is not None and ring_new is not None: 
            # if num==7:plotter.add_mesh(ring, color="white",opacity=0.5)
            # if num==7:plotter.add_mesh(ring_new, color="red",opacity=0.5)
            vectors_center,point_matching_li,point_matching_li_new =calculate_points_dif_2ring(ring,ring_new, ring_centers[num][ring_num][1])   
            
            if len(point_matching_li_new)==len(point_matching_li):
                point_matching_li, point_matching_li_new = remove_close_points(point_matching_list_orig, point_matching_list_orig_new, point_matching_li, point_matching_li_new)
                if len(point_matching_li_new)!=0:
                    point_matching_list_orig.append(point_matching_li)
                    point_matching_list_orig_new.append(point_matching_li_new)
                    # if num==7:plotter.add_points(np.array(point_matching_li), color="white")
                    # if num==7:plotter.add_points(np.array(point_matching_li_new), color="red")
    # if num==7:plotter.show(jupyter_backend='static')
    Origin= [item for sublist in point_matching_list_orig_new for item in sublist]
    Target= [item for sublist in point_matching_list_orig for item in sublist]
    target_points.append(Target)
    origin_points.append(Origin)
    deformation_secod[num]=rbf_calculation_def(meshes_new[num],Target,Origin)

In [ ]:
print(number_rings,len(target_points[0]))

In [ ]:
# mesh_T=pv.PolyData(meshes[0].points+deformation_secod[7]+deformation_init[7])
plot_mesh(mesh1=meshes[0],mesh2=Cline,point_last=np.array(target_points[0]))

In [ ]:
#calculate mesh final:

# Perform element-wise addition
deformation_init_temp = np.array(deformation_init[1])
deformation_secod_temp = np.array(deformation_secod[1])
deformation =  deformation_secod_temp+deformation_init_temp


In [ ]:
meshes_new_new = pv.PolyData(meshes[0].points,meshes[0].faces)
meshes_new_new["Displacement"] = deformation

meshes_new_new.fill_holes(50,inplace=True)
deform_0.fill_holes(50,inplace=True)

name= load_name_mesh_load[1]+"_def.vtp"
meshes_new_new.save(name)
